In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

sns.set_theme(style="whitegrid")

df = pd.read_csv("../data/processed/enron_clean.csv")
print("Dimensions :", df.shape)
print("Valeurs manquantes :\n", df.isnull().sum())

# Sécurité
df = df.dropna(subset=["text_clean"]).reset_index(drop=True)

Dimensions : (29766, 2)
Valeurs manquantes :
 text_clean    0
label         0
dtype: int64


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text_clean"], df["label"],
    test_size=0.2, random_state=42, stratify=df["label"]
)

print("Entraînement :", len(X_train), "emails")
print("Test         :", len(X_test), "emails")
print("\nProportion de spam (label 1) :")
print("  Entraînement : {:.2f} %".format(100 * y_train.mean()))
print("  Test         : {:.2f} %".format(100 * y_test.mean()))

Entraînement : 23812 emails
Test         : 5954 emails

Proportion de spam (label 1) :
  Entraînement : 46.95 %
  Test         : 46.94 %


In [3]:
tfidf = TfidfVectorizer(min_df=3, max_df=0.95, max_features=50000)

X_train_tfidf = tfidf.fit_transform(X_train)   # on apprend sur l'entraînement
X_test_tfidf = tfidf.transform(X_test)         # on applique seulement au test

print("Matrice d'entraînement :", X_train_tfidf.shape)
print("Matrice de test        :", X_test_tfidf.shape)
print("Taille du vocabulaire retenu :", len(tfidf.get_feature_names_out()))

remplissage = X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])
print("Proportion de cases non nulles : {:.3f} %".format(100 * remplissage))

Matrice d'entraînement : (23812, 47235)
Matrice de test        : (5954, 47235)
Taille du vocabulaire retenu : 47235
Proportion de cases non nulles : 0.182 %


In [4]:
i = 0
print("Email (début) :", X_train.iloc[i][:300], "...\n")

vec = X_train_tfidf[i].toarray().ravel()
mots = np.array(tfidf.get_feature_names_out())
top = vec.argsort()[::-1][:10]

pd.DataFrame({"mot": mots[top], "poids TF-IDF": vec[top].round(3)})

Email (début) : dabhol power jim meet us tomorrow one associates worked projects dpc visit january name krishnarao pinnnamaneni gone nombre weeks starting wednesday please let shirley crenshaw assistant nombre nombre know time would work tuesday vince james hughes enron development nombre nombre nombre nombre nombr ...



,mot,poids TF-IDF
0,dabhol,0.404
1,fundamentals,0.257
2,develop,0.202
3,vince,0.201
4,jim,0.201
5,disturbed,0.190
6,nombre,0.187
7,meet,0.164
8,project,0.162
9,dpc,0.144


In [5]:
pd.DataFrame({"text_clean": X_train, "label": y_train}).to_csv("../data/processed/train.csv", index=False)
pd.DataFrame({"text_clean": X_test, "label": y_test}).to_csv("../data/processed/test.csv", index=False)
print("train.csv et test.csv sauvegardés")

train.csv et test.csv sauvegardés
